# PSPNet（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
Pyramid Pooling Module（PPM）により広域的な文脈情報（コンテキスト）を集約するセマンティックセグメンテーション手法であるPSPNet（Pyramid Scene Parsing Network）を，PASCAL VOC 2007データセットを用いて実装する．事前学習済みResNet50のdilated convolution化による出力ストライドの維持と，複数スケールのプーリングによって大域的な情報を取り込むPPMの仕組みについて理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．20クラスの物体に対して画素単位のクラスラベルが付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます．画像枚数が少ない理由やデータセットの詳細は`fcn.ipynb`および`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## PSPNet（Pyramid Scene Parsing Network）
PSPNetは，バックボーンが出力する特徴マップに対してPyramid Pooling Module（PPM）を適用し，複数スケールの大域的な文脈情報を集約したうえでクラス分類を行うネットワークです．FCNのようにスキップ接続で浅い層の情報を補うのではなく，同じ特徴マップに対して複数スケールのプーリングを行うことで，物体同士の関係性や画像全体の文脈を考慮したセグメンテーションを可能にします．

### バックボーン（dilated ResNet50）
事前学習済みのResNet50を使用しますが，通常のResNet50は入力に対して出力ストライド32（`conv1`+`maxpool`でストライド4，`layer1`はストライド1のまま，`layer2`・`layer3`・`layer4`がそれぞれストライド2ずつダウンサンプリング）まで解像度が落ちてしまい，そのままでは特徴マップが粗く物体境界の再現性が低下します．

そこでPSPNetでは，`layer3`・`layer4`の最初のブロックのstrideを2から1に変更し，代わりに畳み込みのdilation（隙間を空けて畳み込みを行うAtrous Convolution）を大きくすることで，受容野の大きさを保ったまま出力ストライドを8に抑えます（`layer3`はdilation=2，`layer4`はdilation=4）．dilationを変えても畳み込みのカーネルの重み自体の形状は変化しないため，事前学習済みの重みをそのまま使うことができます．

In [ ]:
def make_dilated(layer, dilation):
    """ResNetのlayer（Bottleneckブロックのnn.Sequential）内のstride/dilationを書き換え，
    出力ストライドを変えずに受容野だけを広げるdilated convolutionへ変換する．
    重みの形状は変わらないため，事前学習済み重みはそのまま利用できる．"""
    first_block = layer[0]
    # 最初のブロックのstrideを1にすることで，このlayerによるダウンサンプリングを無効化する
    if first_block.downsample is not None:
        first_block.downsample[0].stride = (1, 1)
    first_block.conv2.stride = (1, 1)

    for block in layer:
        # 3x3畳み込み（conv2）のdilationを広げ，strideで失った分の受容野を補う
        block.conv2.dilation = (dilation, dilation)
        block.conv2.padding = (dilation, dilation)


class DilatedResNet50Backbone(nn.Module):
    """layer3・layer4をdilated convolution化し，出力ストライドを32から8に変更したResNet50"""
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)  # stride 4
        self.layer1 = resnet.layer1  # stride 4のまま
        self.layer2 = resnet.layer2  # stride 8
        self.layer3 = resnet.layer3  # 本来stride16 -> dilation化によりstride8のまま
        self.layer4 = resnet.layer4  # 本来stride32 -> dilation化によりstride8のまま

        make_dilated(self.layer3, dilation=2)
        make_dilated(self.layer4, dilation=4)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)  # 出力チャンネル2048，出力ストライド8
        return x


# 出力ストライドが8になっている（32ではない）ことを形状で確認する
_backbone_check = DilatedResNet50Backbone().to(device)
with torch.no_grad():
    _out = _backbone_check(torch.zeros(1, 3, 256, 256).to(device))
print('backbone output shape:', _out.shape)  # (1, 2048, 32, 32) であれば出力ストライド8
assert _out.shape[-1] == 256 // 8
del _backbone_check, _out

### Pyramid Pooling Module（PPM）
バックボーンが出力する特徴マップ（出力ストライド8）に対して，`nn.AdaptiveAvgPool2d`で1×1，2×2，3×3，6×6の4種類のスケールにプーリングします．各スケールでプーリングした特徴マップは1×1畳み込みでチャンネル数を削減したのち，バイリニア補間で元の特徴マップと同じ解像度までアップサンプリングします．最後に，これら4つのマルチスケール特徴と元の特徴マップをチャンネル方向にconcatすることで，局所的な情報と大域的な文脈情報の両方を持つ特徴マップを得ます．

In [ ]:
class PyramidPoolingModule(nn.Module):
    def __init__(self, in_channels, pool_sizes=(1, 2, 3, 6)):
        super().__init__()
        out_channels = in_channels // len(pool_sizes)
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(pool_size),
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for pool_size in pool_sizes
        ])

    def forward(self, x):
        h, w = x.shape[2:]
        # 元の特徴マップに，各スケールでプーリング・アップサンプリングした特徴をconcatする
        feats = [x]
        for stage in self.stages:
            feat = stage(x)
            feat = F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)
            feats.append(feat)
        return torch.cat(feats, dim=1)


class PSPNet(nn.Module):
    def __init__(self, n_class=n_class):
        super().__init__()
        self.backbone = DilatedResNet50Backbone()
        self.ppm = PyramidPoolingModule(in_channels=2048, pool_sizes=(1, 2, 3, 6))

        # PPM出力のチャンネル数: 2048（元の特徴） + 512×4（各スケール，2048//4=512） = 4096
        ppm_out_channels = 2048 + (2048 // 4) * 4
        self.classifier = nn.Sequential(
            nn.Conv2d(ppm_out_channels, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(512, n_class, kernel_size=1),
        )

    def forward(self, x):
        input_size = x.shape[2:]
        feat = self.backbone(x)       # 出力ストライド8
        feat = self.ppm(feat)         # マルチスケールの文脈情報を集約
        out = self.classifier(feat)
        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)  # 入力サイズへ復元
        return out

## 損失関数
セグメンテーションは画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用します．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．原論文にある補助損失（`layer3`の出力に対する追加の分類ヘッド）は，実装簡略化のため省略します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
PSPNetを構築し，最適化手法としてAdamを設定します．バックボーンには事前学習済みの重みを使用しているため，スクラッチ学習のモデルよりも小さい学習率を用います．

In [ ]:
model = PSPNet(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてPSPNetを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoUの平均であるmIoU（mean IoU）を求めます．mIoUの算出には`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します。

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのPSPNetとの違い

| 項目 | オリジナル論文 (Zhao et al., 2017) | 本ノートブック |
| :-- | :-- | :-- |
| バックボーン | 事前学習済みResNet（50や101など，dilated convolution化） | 事前学習済みResNet50（同様にdilated convolution化） |
| 補助損失 (Auxiliary Loss) | `layer3`の出力に対する追加の分類ヘッドを持ち，学習を補助 | 実装簡略化のため省略（`layer3`の出力は使用しない） |
| データ拡張 | マルチスケール学習・ランダムな回転など複数の工夫を実施 | 固定サイズへのリサイズのみ（データ拡張なし） |
| 学習データ量 | ADE20K・PASCAL VOC（拡張データSBDを含む）など大規模データ | VOC2007のtrainvalのみ（422枚） |

## 課題

1. Pyramid Pooling Moduleのプーリングスケール（`pool_sizes`）を変更し，mIoUや推論結果がどのように変化するか確認してください．
2. 本ノートブックでは省略した補助損失（`layer3`の出力に対する分類ヘッドとその損失）を追加し，学習の安定性や精度への影響を確認してください．
3. PPMを使わずバックボーンの出力をそのまま`classifier`に入力した場合（PPMなしPSPNet）と比較し，Pyramid Pooling Moduleの効果を確認してください。